# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR^2 mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset Croissant schema URL is:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install `mlcroissant` library if not already available
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show basic metadata
metadata_obj = dataset.metadata
print(f"Dataset name: {getattr(metadata_obj, 'name', None)}\n")
print(f"Description: {getattr(metadata_obj, 'description', None)}\n")

## 2. Data Overview
Review available record sets, their IDs, and the fields under each record set. This will help us understand the data structure before extracting the records.

We will enumerate all record sets by their `@id` (as required by Croissant semantics), and show their available fields and columns.

In [ ]:
# Enumerate and review all record sets, fields, and columns by their @id
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets found in the metadata.")
else:
    for idx, record_set in enumerate(record_sets):
        print(f"Record set {idx+1} @id: {record_set.id}")
        print(f"  Name: {getattr(record_set, 'name', None)}")
        fields = getattr(record_set, 'fields', [])
        if fields:
            print("  Fields:")
            for f in fields:
                print(f"    @id: {f.id} - Name: {getattr(f, 'name', None)} - dataType: {getattr(f, 'data_type', None)}")
        columns = getattr(record_set, 'columns', [])
        if columns:
            print("  Columns:")
            for c in columns:
                print(f"    @id: {c.id} - Name: {getattr(c, 'name', None)} - dataType: {getattr(c, 'data_type', None)}")
        print("-")

## 3. Data Extraction
Load data from the main record set(s) into DataFrame(s) for analysis using the Croissant record set `@id`s and fields/columns as above.

Below, all available record sets are loaded and displayed. Each DataFrame is indexed by its record set `@id` for clarity.

In [ ]:
# Find all record set @id(s)
rs_ids = [rs.id for rs in getattr(dataset.metadata, 'record_sets', [])]
print(f"Record sets available: {rs_ids}\n")

dataframes = {}
for rs_id in rs_ids:
    print(f"Loading records for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if not records:
        print(f"No records for {rs_id}\n")
    else:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Columns for {rs_id}:", df.columns.tolist())
        print(f"First records for {rs_id}:")
        display(df.head())


## 4. Exploratory Data Analysis (EDA)

Let's select the primary record set and a representative numeric field for filtering, normalization, and aggregation. We'll reference all by the correct `@id`, as shown in the metadata above.

In [ ]:
# --- Update these IDs based on cell 2 output if needed ---
# For the demo we will assume a record set and numeric field exists

if dataframes:
    # Use first available record set
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # Try to find the first numeric looking column (float or int type)
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    print(f"Primary record set: {record_set_id}\nNumeric field chosen: {numeric_field_id}")

    if numeric_field_id is not None:
        # Example: filter where numeric field is above threshold
        threshold = df[numeric_field_id].quantile(0.9)  # top 10% values
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical field if available
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and (df[col].dtype.name == 'object' or pd.api.types.is_categorical_dtype(df[col])):
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
    else:
        print("No numeric field found in the primary record set for analysis.")
else:
    print("No dataframes loaded for analysis.")

## 5. Visualization
Visualize distributions of the primary numeric field (using its Croissant `@id` or name as a label).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=40)
    plt.title(f'Distribution of {numeric_field_id} in {record_set_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If a group_field_id was found, show group means
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10, 5))
        order = grouped_df[group_field_id]
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df, order=order)
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=60)
        plt.show()
else:
    print("No suitable field for visualization.")

## 6. Conclusion

- Used `mlcroissant` to load a harmonized FAIR^2 mass spectrometry dataset from a Croissant schema.
- Explored all available record sets by their `@id`, and extracted their data into DataFrames.
- Selected numeric fields by their Croissant `@id`/column name for filtering, normalization, and grouping.
- Visualized distributions and summarized grouped data to illustrate typical exploratory analysis steps.

This approach can be generalized to other Croissant datasets, always referring to record sets and fields by their unique `@id` for reproducibility and schema consistency.